# Curriculum-Industry Skill Feature Store Using Feast

## MLOps / Feast Skill-Gap Assignment

**Objective:** Convert the self-created CSE curriculum-industry skill-gap dataset into a simple Feast-based feature store and demonstrate feature engineering, Entity, Data Source, FeatureView, `feast apply`, historical retrieval, materialization, online retrieval, and a simple ML prediction.

### End-to-end flow

**Original Dataset → Feature Engineering → Parquet Offline Data → Feast FeatureView → Historical Features / Materialization → Model Training / Online Store → Online Retrieval → Prediction**


# 1. Install Required Libraries

In [ ]:
!pip -q install feast==0.64.0 pyarrow pandas numpy scikit-learn joblib


# 2. Import Libraries

In [ ]:
import os
import shutil
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import feast
from feast import FeatureStore

print("Feast version:", feast.__version__)


# 3. Upload and Load My Unique Dataset

Upload exactly:

`CSE_Employability_Skill_Gap_Unique_Dataset.csv`


In [ ]:
from google.colab import files

uploaded = files.upload()
data = pd.read_csv("CSE_Employability_Skill_Gap_Unique_Dataset.csv")

print("Shape:", data.shape)
display(data.head())


# 4. Dataset Inspection

In [ ]:
print(data.info())
print("\nMissing values:")
display(data.isnull().sum().to_frame("missing"))

print("\nDuplicate rows:", data.duplicated().sum())
print("Duplicate Student IDs:", data["Student_ID"].duplicated().sum())


# 5. Basic Data Cleaning

In [ ]:
data = data.drop_duplicates().copy()
data = data.drop_duplicates(subset=["Student_ID"], keep="first").copy()

for col in ["Skill_Gap_Category", "Priority_Training_Area"]:
    data[col] = data[col].astype(str).str.strip()

print("Cleaned shape:", data.shape)


# 6. Feature Engineering

The feature dataset is intentionally different from the original dataset.

### Engineered Feast features

For each skill, the notebook calculates:

**Skill Gap = Industry Requirement − Curriculum Score**

It also creates:

- Average Curriculum Score
- Average Industry Requirement
- Overall Gap Score
- Skill Readiness Score
- Strongest Skill
- Weakest Skill

The target remains `Skill_Gap_Category`.


In [ ]:
skills = {
    "Programming": ("Programming_Curriculum", "Programming_Industry"),
    "Database": ("Database_Curriculum", "Database_Industry"),
    "Cloud": ("Cloud_Curriculum", "Cloud_Industry"),
    "DataAnalysis": ("DataAnalysis_Curriculum", "DataAnalysis_Industry"),
    "ProblemSolving": ("ProblemSolving_Curriculum", "ProblemSolving_Industry"),
    "Communication": ("Communication_Curriculum", "Communication_Industry"),
    "Teamwork": ("Teamwork_Curriculum", "Teamwork_Industry"),
    "Aptitude": ("Aptitude_Curriculum", "Aptitude_Industry"),
}

feature_df = data.copy()

for skill, (curr_col, ind_col) in skills.items():
    feature_df[f"{skill}_Gap"] = (
        feature_df[ind_col] - feature_df[curr_col]
    ).round(2)

curr_cols = [v[0] for v in skills.values()]
ind_cols = [v[1] for v in skills.values()]
gap_cols = [f"{s}_Gap" for s in skills]

feature_df["Average_Curriculum_Score"] = feature_df[curr_cols].mean(axis=1).round(2)
feature_df["Average_Industry_Requirement"] = feature_df[ind_cols].mean(axis=1).round(2)
feature_df["Calculated_Overall_Gap"] = feature_df[gap_cols].mean(axis=1).round(2)
feature_df["Skill_Readiness_Score"] = np.clip(
    10 - feature_df["Calculated_Overall_Gap"], 0, 10
).round(2)

feature_df["Strongest_Skill"] = feature_df[curr_cols].idxmax(axis=1)
feature_df["Weakest_Skill"] = feature_df[curr_cols].idxmin(axis=1)

display(feature_df.head())


# 7. Prepare Feast Feature Data

Feast needs:

- an entity key
- an event timestamp
- a created timestamp
- feature columns

The entity in this implementation is **Student** and the join key is `student_id`.


In [ ]:
feature_df["student_id"] = feature_df["Student_ID"].astype(str)

base_time = pd.Timestamp("2026-01-01", tz="UTC")
feature_df["event_timestamp"] = (
    base_time + pd.to_timedelta(np.arange(len(feature_df)), unit="s")
)
feature_df["created_timestamp"] = (
    feature_df["event_timestamp"] + pd.Timedelta(seconds=1)
)

feast_feature_columns = [
    "student_id",
    "event_timestamp",
    "created_timestamp",
    "Year",
    "Semester",
    "Attendance_Percent",
    "Projects_Completed",
    "Certifications",
    "Average_Curriculum_Score",
    "Average_Industry_Requirement",
    "Calculated_Overall_Gap",
    "Skill_Readiness_Score",
] + gap_cols

# Add the curriculum/industry scores as additional features.
feast_feature_columns += curr_cols + ind_cols

feast_data = feature_df[feast_feature_columns].copy()

print("Feast feature dataset shape:", feast_data.shape)
display(feast_data.head())


# 8. Save Parquet Offline Data

In [ ]:
repo_path = "/content/cse_employability_feast"

if os.path.exists(repo_path):
    shutil.rmtree(repo_path)

os.makedirs(f"{repo_path}/data", exist_ok=True)

feast_data.to_parquet(
    f"{repo_path}/data/student_skill_features.parquet",
    index=False
)

print("Saved Parquet feature data.")


# 9. Create Feast Configuration

In [ ]:
feature_store_yaml = '''
project: cse_employability_project
registry: data/registry.db
provider: local

offline_store:
  type: file

online_store:
  type: sqlite
  path: data/online_store.db
'''

with open(f"{repo_path}/feature_store.yaml", "w") as f:
    f.write(feature_store_yaml)

print("feature_store.yaml created.")


# 10. Define Feast Entity, Data Source, FeatureView and FeatureService

In [ ]:
feature_definition = '''
from datetime import timedelta

from feast import Entity, FeatureView, FeatureService, Field, FileSource
from feast.types import Float32, Int64

student = Entity(
    name="student",
    join_keys=["student_id"],
    description="CSE graduate student"
)

student_source = FileSource(
    name="student_skill_source",
    path="data/student_skill_features.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp"
)

student_skill_features = FeatureView(
    name="student_skill_features",
    entities=[student],
    ttl=timedelta(days=3650),
    schema=[
        Field(name="Year", dtype=Int64),
        Field(name="Semester", dtype=Int64),
        Field(name="Attendance_Percent", dtype=Float32),
        Field(name="Projects_Completed", dtype=Int64),
        Field(name="Certifications", dtype=Int64),

        Field(name="Average_Curriculum_Score", dtype=Float32),
        Field(name="Average_Industry_Requirement", dtype=Float32),
        Field(name="Calculated_Overall_Gap", dtype=Float32),
        Field(name="Skill_Readiness_Score", dtype=Float32),

        Field(name="Programming_Gap", dtype=Float32),
        Field(name="Database_Gap", dtype=Float32),
        Field(name="Cloud_Gap", dtype=Float32),
        Field(name="DataAnalysis_Gap", dtype=Float32),
        Field(name="ProblemSolving_Gap", dtype=Float32),
        Field(name="Communication_Gap", dtype=Float32),
        Field(name="Teamwork_Gap", dtype=Float32),
        Field(name="Aptitude_Gap", dtype=Float32),

        Field(name="Programming_Curriculum", dtype=Float32),
        Field(name="Programming_Industry", dtype=Float32),
        Field(name="Database_Curriculum", dtype=Float32),
        Field(name="Database_Industry", dtype=Float32),
        Field(name="Cloud_Curriculum", dtype=Float32),
        Field(name="Cloud_Industry", dtype=Float32),
        Field(name="DataAnalysis_Curriculum", dtype=Float32),
        Field(name="DataAnalysis_Industry", dtype=Float32),
        Field(name="ProblemSolving_Curriculum", dtype=Float32),
        Field(name="ProblemSolving_Industry", dtype=Float32),
        Field(name="Communication_Curriculum", dtype=Float32),
        Field(name="Communication_Industry", dtype=Float32),
        Field(name="Teamwork_Curriculum", dtype=Float32),
        Field(name="Teamwork_Industry", dtype=Float32),
        Field(name="Aptitude_Curriculum", dtype=Float32),
        Field(name="Aptitude_Industry", dtype=Float32),
    ],
    source=student_source,
    online=True
)

employability_service = FeatureService(
    name="cse_employability_service",
    features=[student_skill_features]
)
'''

with open(f"{repo_path}/features.py", "w") as f:
    f.write(feature_definition)

print("Feast definitions written to features.py")


# 11. Register Features Using `feast apply`

This registers the Entity, Data Source/FeatureView definitions, and FeatureService with Feast.


In [ ]:
%cd /content/cse_employability_feast
!feast apply


# 12. Verify the Feast Objects

In [ ]:
!feast entities list
!feast feature-views list


# 13. Create FeatureStore Object

In [ ]:
store = FeatureStore(repo_path=repo_path)
service = store.get_feature_service("cse_employability_service")

print("FeatureStore is ready.")


# 14. Historical Feature Retrieval

Historical retrieval demonstrates point-in-time feature retrieval from the offline store.


In [ ]:
# Keep target separately for model training.
target_df = feature_df[[
    "student_id",
    "event_timestamp",
    "Skill_Gap_Category"
]].copy()

historical = store.get_historical_features(
    entity_df=target_df,
    features=service
).to_df()

print("Historical feature shape:", historical.shape)
display(historical.head())


# 15. Train a Simple ML Model Using Feast Historical Features

The model uses the features returned by Feast rather than manually rebuilding the feature dataset at training time.


In [ ]:
feast_model_features = [
    "Year",
    "Semester",
    "Attendance_Percent",
    "Projects_Completed",
    "Certifications",
    "Average_Curriculum_Score",
    "Average_Industry_Requirement",
    "Calculated_Overall_Gap",
    "Skill_Readiness_Score",
] + gap_cols

X = historical[feast_model_features].copy()
y = historical["Skill_Gap_Category"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000))
])

model.fit(X_train, y_train)

pred = model.predict(X_test)
accuracy = accuracy_score(y_test, pred)

print("Model accuracy:", round(accuracy * 100, 2), "%")
print("\nClassification report:")
print(classification_report(y_test, pred))


# 16. Materialize Features into the Online Store

In [ ]:
start_time = feast_data["event_timestamp"].min().strftime("%Y-%m-%dT%H:%M:%S")
end_time = (
    feast_data["event_timestamp"].max() + pd.Timedelta(minutes=1)
).strftime("%Y-%m-%dT%H:%M:%S")

print("Start:", start_time)
print("End:", end_time)

!feast materialize $start_time $end_time


# 17. Online Feature Retrieval

Retrieve the latest stored feature values for one student.


In [ ]:
sample_student = feature_df.iloc[0]["student_id"]

online = store.get_online_features(
    features=service,
    entity_rows=[{"student_id": sample_student}]
).to_dict()

online_df = pd.DataFrame(online)

display(online_df)


# 18. Use Online Feast Features for a Final Prediction

In [ ]:
# Convert online output into the exact model feature order.
online_row = pd.DataFrame({
    col: [online_df[col].iloc[0]]
    for col in feast_model_features
})

final_prediction = model.predict(online_row)[0]

print("Student:", sample_student)
print("Final predicted skill-gap category:", final_prediction)


# 19. Required Analysis — Assignment Answers

### 1. What is the entity?
The entity is **Student**, with `student_id` as the join key.

### 2. What features are stored in the FeatureView?
Academic indicators, curriculum scores, industry requirements, individual skill gaps, average scores, calculated overall gap, and skill-readiness score are stored.

### 3. Explain how one feature was calculated.
For example:

`Programming_Gap = Programming_Industry - Programming_Curriculum`

A positive value indicates that the student's industry requirement is higher than the curriculum score.

### 4. Difference between original dataset and feature dataset
The original dataset contains the student records and project-level information. The Feast feature dataset contains the engineered numerical features required for ML and includes an entity key plus event timestamps.

### 5. Purpose of the offline store
The offline store keeps historical feature data used for training and historical feature retrieval.

### 6. Purpose of the online store
The online store keeps materialized feature values that can be retrieved quickly for prediction.

### 7. Purpose of `feast apply`
`feast apply` registers the Feast entities, feature views and related definitions in the feature registry.

### 8. What does materialization do?
Materialization loads feature values from the historical/offline source into the online store for online retrieval.

### 9. Advantage of Feast
Feast provides a consistent feature definition and retrieval mechanism between training and prediction, reducing the need to manually recreate features separately.

### 10. Two limitations
1. The dataset is synthetic and may not represent real student performance.
2. Industry requirements are represented by simulated values and may not fully reflect current hiring-market requirements.

### 11. Two possible improvements
1. Add verified industry skill requirements from multiple companies and job descriptions.
2. Add real student assessment data and periodically update the feature store.


# 20. Results to Report

The final submission should include:

- Historical feature output
- Model accuracy
- Online feature output
- One final prediction

Run the cells above and capture these outputs for the README/report.


# 21. Final Architecture

```text
Original Skill-Gap Dataset
          |
          v
   Feature Engineering
          |
          v
 Parquet Offline Feature Data
          |
          v
      Feast Entity
          |
          v
       FeatureView
          |
          +----------------------+
          |                      |
          v                      v
Historical Retrieval       Materialization
          |                      |
          v                      v
   Model Training          Online Store
                                 |
                                 v
                         Online Retrieval
                                 |
                                 v
                             Prediction
```
